# Rekurentiniai neuroniniai tinklai

Ankstesniame modulyje aptarėme turtingas semantines teksto reprezentacijas. Architektūra, kurią naudojome, fiksuoja žodžių sakinyje apibendrintą reikšmę, tačiau ji neatsižvelgia į žodžių **tvarką**, nes po įterpimo atliekama agregavimo operacija pašalina šią informaciją iš originalaus teksto. Kadangi šie modeliai negali atvaizduoti žodžių eiliškumo, jie negali išspręsti sudėtingesnių arba dviprasmiškų užduočių, tokių kaip teksto generavimas ar klausimų atsakymas.

Norėdami fiksuoti teksto sekos reikšmę, naudosime neuroninio tinklo architektūrą, vadinamą **rekurentiniu neuroniniu tinklu** arba RNN. Naudojant RNN, mūsų sakinys perduodamas tinklu po vieną ženklą, o tinklas sukuria tam tikrą **būseną**, kurią tada perduodame tinklui kartu su kitu ženklu.

![Vaizdas, rodantis rekursyvinio neuroninio tinklo generavimo pavyzdį.](../../../../../translated_images/lt/rnn.27f5c29c53d727b5.webp)

Turint įvesties ženklų seką $X_0,\dots,X_n$, RNN sukuria neuroninių tinklų blokų seką ir treniruoja šią seką iš karto naudojant atgalinį sklidimą. Kiekvienas tinklo blokas gauna porą $(X_i,S_i)$ kaip įvestį ir duoda $S_{i+1}$ kaip rezultatą. Galutinė būsena $S_n$ arba išvestis $Y_n$ patenka į linijinį klasifikatorių, kuris tampa galutiniu rezultatu. Visi tinklo blokai naudoja tas pačias svorio reikšmes ir treniruojami kartu vienu atgalinio sklidimo žingsniu.

> Aukščiau pateikta iliustracija rodo rekursyvinį neuroninį tinklą atsuktą formą (kairėje) ir kompaktiškesnį rekursyvinį vaizdavimą (dešinėje). Svarbu suprasti, kad visi RNN elementai turi tuos pačius **bendrinamus svorius**.

Kadangi būsena vektoriai $S_0,\dots,S_n$ perduodami tinklu, RNN sugeba išmokti sekos priklausomybes tarp žodžių. Pvz., kai žodis *not* (ne) pasirodo sekoje, tinklas gali išmokti neigti tam tikrus elementus būsenos vektoriuje.

Viduje kiekvienas RNN elementas turi du svorių matricas: $W_H$ ir $W_I$, bei poslinkį $b$. Kiekviename RNN žingsnyje, turint įvestį $X_i$ ir būseną $S_i$, išvesties būsena skaičiuojama kaip $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, kur $f$ yra aktyvavimo funkcija (dažnai $\tanh$).

> Tokiems uždaviniams kaip teksto generavimas (kurį aptarsime kitame skyriuje) ar mašininis vertimas, mes taip pat norime gauti išvesties reikšmę kiekviename RNN žingsnyje. Tokiu atveju yra papildoma matrica $W_O$, o išvestis skaičiuojama kaip $Y_i=f(W_O\times S_i+b_O)$.

Pažiūrėkime, kaip rekursyviniai neuroniniai tinklai gali padėti mums klasifikuoti naujienų duomenų rinkinį.

> Norint pradėti darbą smėlio dėžės aplinkoje, turime paleisti šią ląstelę, kad įsitikintume, jog reikiama biblioteka įdiegta ir duomenys yra paruošti iš anksto. Jei dirbate vietoje, galite praleisti šią ląstelę.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Treniruojant didelius modelius, gali kilti problemų dėl GPU atminties paskirstymo. Taip pat gali prireikti eksperimentuoti su skirtingais mažųjų partijų dydžiais, kad duomenys tilptų į mūsų GPU atmintį, tačiau treniruotė būtų pakankamai greita. Jeigu vykdote šį kodą savo GPU mašinoje, galite eksperimentuoti keisdami mažųjų partijų dydį, kad paspartintumėte treniruotę.

> **Pastaba**: Žinoma, kad tam tikros NVidia tvarkyklių versijos nepašalina atminties po modelio treniravimo. Šiame užrašų knygelėje vykdome kelis pavyzdžius, ir tai gali sukelti atminties išsekimą tam tikrose aplinkose, ypač jei atliekate savo eksperimentus toje pačioje užrašų knygelėje. Jeigu susiduriate su keistomis klaidomis pradėję treniruoti modelį, gali būti naudinga perkrauti užrašų knygelės branduolį.


In [3]:
batch_size = 16
embed_size = 64

## Paprastas RNN klasifikatorius

Paprasto RNN atveju kiekviena pasikartojanti vienetė yra paprastas linijinis tinklas, kuris priima įvesties vektorių ir būsenos vektorių, ir generuoja naują būsenos vektorių. Keras tai galima pavaizduoti sluoksniu `SimpleRNN`.

Nors galime tiesiogiai perduoti vieno karšto koduoto žodžio žymes RNN sluoksniui, tai nėra gera idėja dėl jų didelės dimensijos. Todėl naudosime įdėjimo sluoksnį, kad sumažintume žodžių vektorių dimensiją, po to RNN sluoksnį ir galiausiai `Dense` klasifikatorių.

> **Pastaba**: Atvejais, kai dimensija nėra tokia didelė, pavyzdžiui naudojant simbolių lygmens tokenizaciją, gali būti prasminga tiesiogiai perduoti vieno karšto koduotas žymes į RNN ląstelę.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Pastaba:** Čia mes naudojame nemokytą įterpimo sluoksnį paprastumui, bet geresniems rezultatams galime naudoti iš anksto apmokytą įterpimo sluoksnį naudojant Word2Vec, kaip aprašyta ankstesniame skyriuje. Būtų geras pratimas jums adaptuoti šį kodą taip, kad jis veiktų su iš anksto apmokytais įterpimais.

Dabar apmokysime mūsų RNN. RNN apskritai yra gana sunku apmokyti, nes kai RNN ląstelės yra išskleistos per sekos ilgį, gautas sluoksnių skaičius, dalyvaujančių atgalinės sklaidos procese, yra gana didelis. Todėl turime pasirinkti mažesnį mokymosi greitį ir apmokyti tinklą didesniu duomenų rinkiniu, kad gautume gerus rezultatus. Tai gali užtrukti gana ilgai, todėl pageidautina naudoti GPU.

Norėdami pagreitinti procesą, apmokysime RNN modelį tik pagal naujienų antraštes, nesinaudodami aprašymu. Galite pabandyti apmokyti su aprašymu ir pažiūrėti, ar modelį pavyks apmokyti.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Atkreipkite dėmesį**, kad tikslumas čia greičiausiai bus mažesnis, nes mes mokomės tik iš naujienų pavadinimų.


## Kintamųjų sekų peržiūra 

Atminkite, kad sluoksnis `TextVectorization` automatiškai užpildo kintamo ilgio sekas minibatche su užpildo žetonais. Paaiškėja, kad tie žetonai taip pat dalyvauja mokyme ir gali apsunkinti modelio konvergenciją.

Yra keli būdai, kaip galime sumažinti užpildymo kiekį. Vienas iš jų – pertvarkyti duomenų rinkinį pagal sekos ilgį ir sugrupuoti visas sekas pagal dydį. Tai galima padaryti naudojant `tf.data.experimental.bucket_by_sequence_length` funkciją (žr. [dokumentaciją](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Kitas būdas – naudoti **maskavimą**. Keras kai kurios sluoksniai palaiko papildomą įvestį, rodančią, kurie žetonai turėtų būti įtraukti mokymo metu. Kad įtrauktume maskavimą į mūsų modelį, galime įtraukti atskirą `Masking` sluoksnį ([dokumentacija](https://keras.io/api/layers/core_layers/masking/)) arba galime nurodyti `mask_zero=True` parametrą mūsų `Embedding` sluoksnyje.

> **Pastaba**: Šis mokymas užtruks apie 5 minutes, kad vienas epochas būtų užbaigta visame duomenų rinkinyje. Galite bet kada nutraukti mokymą, jei trūksta kantrybės. Taip pat galite apriboti duomenų kiekį, naudojamą mokymui, pridėdami `.take(...)` sąlygą po `ds_train` ir `ds_test` duomenų rinkinių.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Dabar, kai naudojame maskavimą, galime apmokyti modelį visam pavadinimų ir aprašymų duomenų rinkiniui.

> **Pastaba**: Ar pastebėjote, kad naudojame vektorizatorių, apmokytą tik pagal naujienų pavadinimus, o ne visą straipsnio tekstą? Tai gali reikšti, kad kai kurie žodžiai bus ignoruojami, todėl geriausia būtų permokyti vektorizatorių. Vis dėlto, tai gali turėti tik nedidelį poveikį, tad paprastumo dėlei tęsime naudoti ankstesnį iš anksto apmokytą vektorizatorių.


## LSTM: Ilgos trumpos atminties tinklas

Viena pagrindinių RNN problemų yra **gradiento nykimas**. RNN gali būti gana ilgi, ir jiems gali būti sunku perduoti gradientus atgal iki pirmojo tinklo sluoksnio atgalinės sklaidos metu. Kai tai nutinka, tinklas negali išmokti ryšių tarp tolimų žetonų. Vienas būdas išvengti šios problemos yra įvesti **aiškią būsenos valdymą** naudojant **vartus**. Du dažniausiai vartus įvedantys architektūros tipai yra **ilgos trumpos atminties tinklas** (LSTM) ir **vartų perdavimo vienetas** (GRU). Čia aptarsime LSTM.

![Paveikslėlis, vaizduojantis ilgos trumpos atminties ląstelės pavyzdį](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

LSTM tinklas organizuotas panašiai kaip RNN, tačiau yra dvi būsenos, kurios perduodamos iš sluoksnio į sluoksnį: tikroji būsena $c$ ir paslėptasis vektorius $h$. Kiekviename vienete paslėptasis vektorius $h_{t-1}$ sujungiama su įvestimi $x_t$, ir kartu jie kontroliuoja, kas nutinka būsenai $c_t$ ir išėjimui $h_{t}$ per **vartus**. Kiekvienas vartas turi sigmoidinę aktyvaciją (išėjimas intervale $[0,1]$), kurią galima laikyti bitine kauke, kai dauginama iš būsenos vektoriaus. LSTM turi šiuos vartus (nuo kairės į dešinę aukščiau esančiame paveikslėlyje):
* **užmiršimo vartas**, kuris nustato, kurias $c_{t-1}$ vektoriaus komponentes reikia pamiršti, o kurias perleisti.
* **įvesties vartas**, kuris nusprendžia, kiek informacijos iš įvesties vektoriaus ir ankstesnio paslėpto vektoriaus turi būti įtrauktas į būsenos vektorių.
* **išėjimo vartas**, kuris pasiima naują būsenos vektorių ir nusprendžia, kurios jo komponentės bus naudojamos naujo paslėpto vektoriaus $h_t$ sukūrimui.

Būsenos $c$ komponentus galima laikyti vėliavėlėmis, kurias galima įjungti arba išjungti. Pavyzdžiui, kai sekoje sutinkame vardą *Alice*, spėjame, kad tai moters vardas, ir įjungiame vėliavą būsenoje, kuri nurodo, kad sakinyje yra moteriškos giminės daiktavardis. Kai vėliau sutinkame žodžius *and Tom*, įjungiame vėliavą, nurodančią, kad turime daugiskaitos daiktavardį. Taip manipuliuodami būsena galime sekti sakinio gramatines savybes.

> **Pastaba**: Štai puikus šaltinis apie LSTM vidinius veikimo principus: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) autorius Christopher Olah.

Nors LSTM ląstelės vidinė struktūra gali atrodyti sudėtinga, Keras slepia šią įgyvendinimą viduje sluoksnyje `LSTM`, tad pavyzdyje aukščiau vienintelis dalykas, kurį reikia padaryti, yra pakeisti rekursinį sluoksnį:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Pastaba**: treniruoti LSTM taip pat yra gana lėtai, ir pradžioje galite nepastebėti daug tikslumo padidėjimo. Gali tekti tęsti treniruotes tam tikrą laiką, kad pasiektumėte gerą tikslumą.


## Dvipusės ir daugiasluoksnės RNN

Mūsų iki šiol pateiktuose pavyzdžiuose pasikartojančios tinklai veikia nuo sekos pradžios iki pabaigos. Tai mums atrodo natūralu, nes tai atitinka tą pačią kryptį, kuria skaitome ar klausomės kalbos. Tačiau scenarijuose, kuriuose reikalingas atsitiktinis įėjimo sekos pasiekimas, prasmingiau atlikti pasikartojančias skaičiavimus abiem kryptimis. RNN, leidžiančios skaičiavimus abiem kryptimis, vadinamos **dvipusėmis** RNN, ir jas galima sukurti apvyniojus pasikartojančią sluoksnį su specialiu `Bidirectonal` sluoksniu.

> **Pastaba**: `Bidirectional` sluoksnis sukuria du tos pačios sluoksnio kopijas ir nustato vienos iš jų `go_backwards` ypatybę į `True`, todėl ji juda priešinga seka kryptimi.

Pasikartojantys tinklai, vienpusiai ar dvipusiai, fiksuoja sekos modelius ir saugo juos būsenos vektoriuose arba grąžina kaip išvestį. Kaip ir konvoliuciniai tinklai, galime sudėti dar vieną pasikartojantį sluoksnį po pirmojo, kad sugautume aukštesnio lygio modelius, sudarytus iš žemesnio lygio modelių, kuriuos išskyrė pirmasis sluoksnis. Tai mus nusiveda prie sąvokos **daugiasluoksnė RNN**, kurią sudaro du ar daugiau pasikartojančių tinklų, kur ankstesnio sluoksnio išvestis perduodama kitam sluoksniui kaip įėjimas.

![Paveikslėlis, rodantis daugiasluoksnę ilgalaikės trumpalaikės atminties RNN](../../../../../translated_images/lt/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Paveikslėlis iš [šio nuostabaus įrašo](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3), kurį parašė Fernando López.*

Keras padaro šių tinklų konstravimą paprastą užduotį, nes tereikia pridėti daugiau pasikartojančių sluoksnių prie modelio. Visuose sluoksniuose, išskyrus paskutinįjį, reikia nurodyti parametrą `return_sequences=True`, nes mums reikia, kad sluoksnis grąžintų visas tarpines būsenas, o ne tik galutinę pasikartojančios skaičiavimo būseną.

Sukurkime dviejų sluoksnių dvipusį LSTM mūsų klasifikacijos problemos sprendimui.

> **Pastaba** šis kodas vėl užtrunka gana ilgai, bet suteikia aukščiausią tikslumą, kurį matėme iki šiol. Todėl gal verta palaukti ir pamatyti rezultatą.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN tinklai kitoms užduotims

Iki šiol daugiausia dėmesio skyrėme RNN tinklų naudojimui tekstų sekų klasifikavimui. Tačiau jie gali atlikti ir daug daugiau užduočių, pavyzdžiui, tekstų generavimą ir mašininį vertimą &mdash; šias užduotis aptarsime kitoje dalyje.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
